![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 8 -- Lab 3: Homework University Admissions

Homework. The Hungarian AI Olympiad's 2025 online qualifier asked: given a school leaver's grades, exam scores and background, will they be admitted to university? The target is yes or no, so this is a classification task and the model is the logistic regression of the last slides. The columns are Hungarian; the setup cell renames them to English. Exam columns use `-1` for a subject the student did not take, a trap you must handle.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** `train.csv` (6,885 students with the `admitted` column) and `test.csv` (765 students without it) from the HAIO 2025 repository (`github.com/Hungarian-AI-Olympiad/HAIO-Hungarian-AI-Olympiad`, folder `2025/nyari-online/adatok`), read straight from GitHub. The official score is ROC-AUC (Day 09); here you use accuracy, the share of correct answers.

---
# Setup

In [ ]:
# pandas, matplotlib and scikit-learn are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q pandas matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [ ]:
BASE = "https://raw.githubusercontent.com/Hungarian-AI-Olympiad/HAIO-Hungarian-AI-Olympiad/main/2025/nyari-online/adatok/"
EN = {"Életkor": "age", "Nem": "sex", "Osztályzat_9": "grade_9", "Osztályzat_10": "grade_10", "Osztályzat_11": "grade_11",
      "Osztályzat_12": "grade_12", "Történelem": "history", "Matematika": "math", "Magyar Nyelv és Irodalom": "hungarian",
      "Informatika": "informatics", "Biológia": "biology", "Fizika": "physics", "Angol": "english", "Német": "german",
      "Informatika_emelt": "informatics_adv", "Biológia_emelt": "biology_adv", "Fizika_emelt": "physics_adv",
      "Angol_emelt": "english_adv", "Német_emelt": "german_adv", "Matematika_emelt": "math_adv", "Történelem_emelt": "history_adv",
      "Magyar Nyelv és Irodalom_emelt": "hungarian_adv", "Szülői Végzettség": "parent_education",
      "Középiskola Presztízse": "school_prestige", "Extrakurrikuláris Tevékenységek": "extracurricular",
      "Tanulási Szokások": "study_habits", "Munkatapasztalat": "work_experience", "Ajánlások Száma": "recommendations",
      "Versenyeken Való Részvétel": "competitions", "Vármegye": "county", "Felvételi Eredmény": "admitted", "ID": "ID"}
train = pd.read_csv(BASE + "train.csv").rename(columns=EN)
test = pd.read_csv(BASE + "test.csv").rename(columns=EN)
print(train.shape, test.shape)
print(list(train.columns))

---
# Part 1 -- EDA with the target in mind

Before any model: how common is 'admitted', what do the `-1` values mean, and which columns actually differ between the two groups?

## Task 1: How many are admitted?

Print `train["admitted"].value_counts(normalize=True)`. Store the share of the bigger class in `majority` (expected about 0.702): a model that always says 'not admitted' gets this accuracy, and that is the number to beat.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(majority - 0.702) < 0.005, 'majority: the larger share in value_counts(normalize=True)'
print('Task 1 passed')

## Task 2: Where are the -1 values?

Compute `(train == -1).sum()` and print only the columns where the count is above 0. Ten columns have them, always in pairs (a subject and its `_adv` flag). What does `-1` mean here?

In [ ]:
# Your code here


## Task 3: What differs between admitted and not?

Print `train.groupby("admitted")[["grade_12", "math", "informatics", "school_prestige", "competitions"]].mean().round(2)`. Which columns barely move between the two groups, and which move a lot?

In [ ]:
# Your code here


## Task 4: Admission rate by county

Compute the mean of `admitted` per `county` with `groupby`, sort it, and print it. Expected: from about 0.16 in the lowest county to 0.625 in Budapest. The county is text with no order, so it will need one-hot encoding.

In [ ]:
# Your code here


---
# Part 2 -- Logistic regression, step by step

Same recipe as Lab 1, with two changes: the model is `LogisticRegression`, the score is `accuracy_score`, and the features are scaled first, because logistic regression is trained by gradient descent and columns on wildly different scales make it slow or stop it from converging.

## Task 5: The scoring function

Write `fit_and_score(X)` that splits `X` and `y = train["admitted"].astype(int)` with `train_test_split(X, y, test_size=0.2, random_state=42)`, fits a `StandardScaler` on `X_train` only, transforms both parts, fits `LogisticRegression(max_iter=1000)`, prints the accuracy on the validation part and returns the model. Test it on the four grade columns. Expected accuracy 0.705: no better than the majority guess, exactly what Task 3 predicted.

Syntax hint (the shape of the call, not the answer):

```python
scaler = StandardScaler().fit(X_train)          # learn mean and std on the training rows only
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
```

In [ ]:
# Your code here


## Task 6: All numeric columns, and why scaling is not optional

Build `numeric`, the list of all columns except `county`, `admitted` and `ID` (29 columns). First fit a `LogisticRegression(max_iter=1000)` on the raw, unscaled `train[numeric]` with the same split and print its accuracy: read the warning scikit-learn prints. Then run `fit_and_score(train[numeric])`. Expected: about 0.850 unscaled with a convergence warning, 0.859 scaled without one.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(numeric) == 29, 'numeric: every column except county, admitted and ID'
print('Task 6 passed')

## Task 7: Handle the -1 values

For the five optional subjects (`informatics`, `biology`, `physics`, `english`, `german`): add a 0/1 column `<subject>_taken` that is 1 when the score is not `-1`, then replace `-1` by 0 in the score column and in its `_adv` column. Do it on a copy `tf = train.copy()`. Build `feats`, the list of all columns of `tf` except `county`, `admitted`, `ID` (34 columns), and run `fit_and_score(tf[feats])`. Expected: still about 0.859. Not every fix moves the score; this one makes the columns mean what they say, which matters when you read the weights.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(feats) == 34 and (tf[feats] == -1).sum().sum() == 0, 'feats: 29 columns plus 5 _taken flags, and no -1 left anywhere'
print('Task 7 passed')

## Task 8: One-hot encode the county

`county` has 20 values and no order. Build `county = pd.get_dummies(tf["county"], prefix="county", dtype=int)`, then `X = pd.concat([tf[feats], county], axis=1)` (54 columns) and run `fit_and_score(X)`. Expected accuracy about 0.871. Store the model as `best`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert X.shape == (6885, 54), 'X: 34 feature columns plus 20 one-hot county columns'
print('Task 8 passed')

## Task 9: Read the weights

`best.coef_[0]` holds one weight per column of `X`, on the scaled features, so they are comparable. Put them in `pd.Series(best.coef_[0], index=X.columns).sort_values()` and print the three most negative and the three most positive. Do they agree with the group means of Task 3 and the county rates of Task 4?

In [ ]:
# Your code here


## Task 10: Probabilities, not just labels

Redo the split and the scaler of `fit_and_score` on `X` by hand, then print `best.predict_proba(...)[:5, 1]` for the first five validation students next to their true `y_val[:5]` values. A probability near 0.5 is a student the model is unsure about.

In [ ]:
# Your code here


---
# Part 3 -- The submission

The official format is one row per test student with the columns `ID` and `admitted`.

## Task 11: Prepare the test table the same way

Apply the Task 7 fixes to a copy of `test`, one-hot encode its county, and `reindex` the result to `X.columns` with `fill_value=0`. Store it in `X_test` and print its shape (expected `(765, 54)`).

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert X_test.shape == (765, 54) and list(X_test.columns) == list(X.columns), 'X_test: same fixes, one-hot county, reindex to X.columns'
print('Task 11 passed')

## Task 12: Fit on all training rows and write the file

Fit a new `StandardScaler` and `LogisticRegression(max_iter=1000)` on all of `X` and `y`, predict `X_test` (scaled with that scaler), and save `pd.DataFrame({"ID": test["ID"], "admitted": predictions})` to `submission.csv` with `index=False`. Print the share of predicted admissions; it should be near the 0.30 of the training table.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert pd.read_csv('submission.csv').shape == (765, 2), 'submission.csv: columns ID and admitted, 765 rows'
print('Task 12 passed')

---
## Task 13: Reflect

The four grades gave the same accuracy as guessing 'not admitted' for everyone, although grades sound like the obvious predictor. What in Task 3 explained that before you fit anything? Why must the scaler be fitted on the training rows only, and reused (not refitted) on the validation and test rows? Accuracy treats a wrong 'yes' and a wrong 'no' the same; give one reason the organisers might prefer a different score (Day 09 will name it).

*Your answers here*

---
## Conclusion

- **Classification task**: a yes/no target, a majority baseline of 0.70 to beat, and accuracy as the score.
- **EDA first**: the group means told you grades would not help and exam scores would, and the county rates told you the county carried signal.
- **Encoding and scaling**: `-1` became a flag plus a zero, the county became 20 one-hot columns, and scaling let logistic regression converge and made its weights readable.
- **Logistic regression in scikit-learn**: `fit`, `predict`, `predict_proba`, the weights, and a submission in the required format.

---

Made By **Sattam Altwaim**